# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [45]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier,AdaBoostClassifier, GradientBoostingClassifier


In [46]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [47]:
spaceship=spaceship.dropna()

**Perform Train Test Split**

In [48]:
features = spaceship.drop(columns='Transported')
target = spaceship["Transported"]

In [49]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=17)

In [50]:
print(X_train.shape)
print(X_test.shape)

(5284, 13)
(1322, 13)


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [51]:
def cabin_value_handle(X_train):
    X_train['Cabin'] = X_train['Cabin'].str.split('/').str[0]
    return X_train


X_train=cabin_value_handle(X_train)

In [52]:
def drop_pass_id(X_train):
    X_train = X_train.drop(columns=["PassengerId","Name"])
    return X_train

X_train = drop_pass_id(X_train)

In [53]:
def bool_type_conversion(X_train):
    X_train["CryoSleep"] = X_train["CryoSleep"].astype(bool)
    X_train["VIP"] = X_train["VIP"].astype(bool)
    return X_train

X_train = bool_type_conversion(X_train)

In [54]:
categorical_cols = X_train.select_dtypes('object')

In [55]:
categorical_cols.columns.tolist()

['HomePlanet', 'Cabin', 'Destination']

In [56]:
def one_hot_encoding(X_train):
    X_train_encoded = encoder.transform(X_train[categorical_cols.columns.tolist()])
    df=pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(categorical_cols.columns.tolist()))
    X_train = X_train.reset_index(drop=True)
    df = df.reset_index(drop=True)
    df = pd.concat([X_train, df], axis=1).reset_index(drop=True)
    df = df.drop(columns=['HomePlanet', 'Cabin', 'Destination'])
    print(df.shape)
    return df

encoder = OneHotEncoder(sparse_output = False)
encoder.fit(X_train[categorical_cols.columns.tolist()])
X_train_encoded = one_hot_encoding(X_train)

(5284, 22)


In [57]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors = 10)

In [58]:
X_train_encoded

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,...,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,True,22.0,False,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,True,42.0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,False,8.0,False,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,False,13.0,False,578.0,77.0,0.0,306.0,5.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,False,35.0,False,102.0,780.0,6.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5279,True,14.0,False,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
5280,False,36.0,True,530.0,5085.0,1.0,146.0,872.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5281,True,28.0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5282,False,18.0,False,0.0,0.0,699.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [59]:
knn.fit(X_train_encoded,y_train)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [60]:
X_test = cabin_value_handle(X_test)
X_test = drop_pass_id(X_test)
X_test = bool_type_conversion(X_test)
X_test = one_hot_encoding(X_test)

(1322, 22)


In [61]:
knn.score(X_test, y_test)

0.7987897125567323

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [62]:
bagging_class = BaggingClassifier(DecisionTreeClassifier(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

In [63]:
bagging_class.fit(X_train_encoded, y_train)

,estimator,DecisionTreeC...(max_depth=20)
,n_estimators,100
,max_samples,1000
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,None
,verbose,0


In [64]:
pred = bagging_class.predict(X_test)

print("R2 score", bagging_class.score(X_test, y_test))

R2 score 0.8071104387291982


- Random Forests

In [65]:
forest = RandomForestClassifier(n_estimators=100,
                             max_depth=20)

In [66]:
forest.fit(X_train_encoded, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [67]:
pred = forest.predict(X_test)

print("R2 score", forest.score(X_test, y_test))

R2 score 0.8055975794251135


- Gradient Boosting

In [68]:
gb_class = GradientBoostingClassifier(max_depth=20,
                                   n_estimators=100)

In [69]:
gb_class.fit(X_train_encoded, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,20
,min_impurity_decrease,0.0
,init,None


In [70]:
pred = gb_class.predict(X_test)

print("R2 score", gb_class.score(X_test, y_test))

R2 score 0.7927382753403933


- Adaptive Boosting

In [71]:
ada_class = AdaBoostClassifier(DecisionTreeClassifier(max_depth=20),
                            n_estimators=100)

In [72]:
ada_class.fit(X_train_encoded, y_train)

,estimator,DecisionTreeC...(max_depth=20)
,n_estimators,100
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,None
,criterion,'gini'
,splitter,'best'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [73]:
pred = ada_class.predict(X_test)

print("R2 score", ada_class.score(X_test, y_test))

R2 score 0.783661119515885


Which model is the best and why?

best-performing model overall was the BaggingClassifier:

BaggingClassifier accuracy: 0.8071

knn classifier: 0.7987

Random Forest classifier R²: 0.8055

Gradient Boosting classifier R²: 0.7927

AdaBoost classifier R²: 0.7836